# ZooKeeperの更新

ZooKeeperを1台ずつ更新する。対象ごとに作業コピーを作り、復帰を確認してから次のサーバーへ進む。

## *Operation Note*

*This is a cell for your own recording. ここに経緯を記述*

# 操作対象クラスタの設定

D10で作成したinventoryとgroup_varsを使用する。

In [ ]:
target_group = 'hadoop_all_testcluster'
target_group

In [ ]:
!ansible {target_group} -m ping

In [ ]:
import yaml
with open('group_vars/' + target_group) as configuration:
    kerberos_enabled = 'kerberos_realm' in yaml.safe_load(configuration)
kerberos_enabled

# 管理操作の認証

Kerberos構成の場合、HDFSの管理操作を行うホストでhdfsユーザーのkeytabを使って認証する。

In [ ]:
if kerberos_enabled:
    !ansible 'hadoop_namenode:hadoop_client' -l {target_group} --become --become-user hdfs -m shell -a 'kinit -kt /etc/hadoop/conf/hdfs.keytab hdfs/$(hostname -f)'

# Playbookの準備

In [ ]:
import os
import tempfile
work_dir = tempfile.mkdtemp()
work_dir

使用するNotebookと同じ版のPlaybookを取得する。

In [ ]:
!git clone https://github.com/NII-cloud-operation/Literate-computing-Hadoop.git {work_dir}/hadoop
!git -C {work_dir}/hadoop log -1

In [ ]:
playbook_dir = os.path.join(work_dir, 'hadoop/playbooks')
playbook_dir

In [ ]:
zookeeper_host = 'XXX.XXX.XXX.70'
zookeeper_host

# Ensembleの確認

In [ ]:
!ansible hadoop_zookeeperserver --become --become-user zookeeper -a 'zkServer.sh --config /etc/zookeeper/conf status' -l {target_group}

全台が応答し、Leaderが1台であることを確認する。過半数を維持するため、Followerから1台ずつ更新する。

# 対象サーバーの更新

停止時にもcurrentのスクリプトを使うため、サービスを停止してから切り替える。

In [ ]:
!ansible {zookeeper_host} --become -m service -a "name=zookeeper-server state=stopped" -l {target_group}

In [ ]:
!ansible-playbook -CDv {playbook_dir}/switch_upgrade_package.yml --tags zookeeper -l {zookeeper_host}

対象1台のZooKeeperのcurrentだけが更新されることを確認する。

In [ ]:
!ansible-playbook {playbook_dir}/switch_upgrade_package.yml --tags zookeeper -l {zookeeper_host}

In [ ]:
!ansible {zookeeper_host} --become -m service -a "name=zookeeper-server state=started" -l {target_group}

In [ ]:
!ansible {zookeeper_host} -m wait_for -a "host={zookeeper_host} port=2181 timeout=120" -l {target_group}

In [ ]:
!ansible {zookeeper_host} --become --become-user zookeeper -a "zkServer.sh --config /etc/zookeeper/conf status" -l {target_group}

LeaderまたはFollowerとして参加したことを確認する。起動中なら再確認する。

In [ ]:
!ansible {zookeeper_host} --become -m package -a "name=curl state=present" -l {target_group}

In [ ]:
!ansible {zookeeper_host} -m shell -a 'curl --fail --no-progress-meter http://localhost:8080/commands/srvr' -l {target_group}

versionが更新先、server_stateがleaderまたはfollowerであることを確認する。AdminServerのポートは対象の設定に合わせる。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState' -l {target_group}

HAがActive/Standbyに戻ったことを確認して次のサーバーへ進む。全台が完了したら、[NameNodeの更新](O12g_Hadoop%20-%20Rolling%20upgrade%20NameNodes.ipynb)へ進む。